# Stage 0: Train VITS TTS on Common Voice Swahili

**GPU Budget:** ~4 hours on Colab T4 / Kaggle T4

This notebook fine-tunes a VITS text-to-speech model on Common Voice Swahili
so we can synthesize Swahili speech from MADLAD translations (En->Sw direction).

**For Sw->En direction**, we use a pretrained English TTS (no training needed).

### Pipeline position
```
Stage 0 (this notebook): Train Swahili TTS
  -> Used by: synthesize_tts.py (generates Sw audio from En->Sw translations)
  -> Feeds into: Stage 3 S2ST training (parallel speech pairs)
```

### Strategy
We fine-tune `facebook/mms-tts-swh` (Meta's Massively Multilingual Speech TTS for Swahili)
on Common Voice Swahili data. This gives us:
- A strong Swahili TTS baseline (MMS already speaks Swahili)
- Adaptation to Common Voice speaker characteristics
- Much faster training than training VITS from scratch

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torchaudio soundfile scipy
# Optional: Coqui TTS for richer training (uncomment if desired)
# !pip install -q TTS

In [ ]:
import os
import sys
import torch

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Clone repo (or upload)
REPO_DIR = '/content/hibiki-sw'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/gambigivesyouwiings/Mambo.git {REPO_DIR}
sys.path.insert(0, REPO_DIR)

# Output directories (on Drive for persistence)
BASE_DIR = '/content/drive/MyDrive/hibiki-sw'
VITS_DIR = f'{BASE_DIR}/vits_sw'
os.makedirs(VITS_DIR, exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Download Common Voice Swahili Dataset

Download the 20GB Common Voice Swahili archive from Mozilla Data Collective.
You need your **client ID** and **access key** from:
https://datacollective.mozillafoundation.org/datasets/cmn3ailbd008nmb07mjyu3xro

After extraction you'll have:
```
/content/cv-corpus-19.0-2024-09-13/sw/
  clips/           # MP3 audio files
  validated.tsv    # Metadata for validated clips (~300hrs)
  train.tsv        # Training split
  dev.tsv          # Dev split
  test.tsv         # Test split
```

In [ ]:
# === OPTION A: Download via Mozilla Data Collective API ===
# Replace with your actual credentials
CLIENT_ID = 'YOUR_CLIENT_ID_HERE'
ACCESS_KEY = 'YOUR_ACCESS_KEY_HERE'
DATASET_URL = 'https://datacollective.mozillafoundation.org/api/v1/datasets/cmn3ailbd008nmb07mjyu3xro/download'

# Download (this takes ~15-20 min for 20GB)
!curl -L -o /content/cv-sw.tar.gz \
    -H "X-Client-Id: {CLIENT_ID}" \
    -H "X-Access-Key: {ACCESS_KEY}" \
    "{DATASET_URL}"

# Extract
!mkdir -p /content/cv-corpus && tar -xzf /content/cv-sw.tar.gz -C /content/cv-corpus

# Find the extracted directory
!ls /content/cv-corpus/

In [ ]:
# Set the dataset path (adjust if your version differs)
# Look at the output above and set the correct path
CV_SW_DIR = '/content/cv-corpus/cv-corpus-19.0-2024-09-13/sw'  # ADJUST THIS

# Verify the dataset
!ls {CV_SW_DIR}/*.tsv
!echo '---'
!head -2 {CV_SW_DIR}/validated.tsv
!echo '---'
!wc -l {CV_SW_DIR}/validated.tsv

## 2. Prepare VITS Training Data

Convert Common Voice data into VITS format:
- Resample audio to 22050 Hz WAV
- Create metadata CSV (audio_path|speaker_id|text)
- Split into train/val sets

In [ ]:
from data.prepare.train_vits import prepare_vits_dataset

train_csv, val_csv, speaker_map = prepare_vits_dataset(
    dataset_dir=CV_SW_DIR,
    output_dir=VITS_DIR,
    split='validated',
    max_samples=50000,     # Start with 50K samples (~100hrs)
    min_duration=1.0,
    max_duration=15.0,
    val_ratio=0.05,
    sample_rate=22050,
)

print(f'\nTrain CSV: {train_csv}')
print(f'Val CSV: {val_csv}')
print(f'Number of speakers: {len(speaker_map)}')

In [ ]:
# Preview the training data
!head -5 {VITS_DIR}/train.csv
!echo '---'
!wc -l {VITS_DIR}/train.csv {VITS_DIR}/val.csv

## 3. Train VITS (Fine-tune MMS-TTS-Swahili)

Fine-tunes `facebook/mms-tts-swh` on the prepared Common Voice data.

- Only decoder, flow, and duration predictor are fine-tuned (encoder stays frozen)
- This makes training ~3x faster than full fine-tuning
- Checkpoints saved every 5000 steps to Google Drive

**Expected time:** ~3-4 hours on a T4 GPU for 50K samples, 100 epochs.

In [ ]:
from data.prepare.train_vits import train_vits_custom
import json

# Load speaker map
with open(f'{VITS_DIR}/speaker_map.json') as f:
    speaker_map = json.load(f)

# Train! Adjust epochs/batch_size based on your dataset size
train_vits_custom(
    train_csv=f'{VITS_DIR}/train.csv',
    val_csv=f'{VITS_DIR}/val.csv',
    output_dir=VITS_DIR,
    speaker_map=speaker_map,
    num_epochs=100,
    batch_size=16,        # Reduce to 8 if OOM on T4
    learning_rate=2e-4,
    resume_from=None,     # Set to checkpoint path to resume
    sample_rate=22050,
    num_workers=2,
)

In [ ]:
# === RESUME TRAINING (if Colab disconnected) ===
# Uncomment and set the checkpoint path:

# from data.prepare.train_vits import train_vits_custom
# import json
#
# with open(f'{VITS_DIR}/speaker_map.json') as f:
#     speaker_map = json.load(f)
#
# train_vits_custom(
#     train_csv=f'{VITS_DIR}/train.csv',
#     val_csv=f'{VITS_DIR}/val.csv',
#     output_dir=VITS_DIR,
#     speaker_map=speaker_map,
#     num_epochs=100,
#     batch_size=16,
#     learning_rate=2e-4,
#     resume_from=f'{VITS_DIR}/checkpoint_25000.pth',  # ADJUST
#     sample_rate=22050,
#     num_workers=2,
# )

## 4. Test the Trained Model

Generate a few Swahili utterances to verify quality.

In [ ]:
from data.prepare.synthesize_tts import MMS_TTS
import IPython.display as ipd
import numpy as np

# Load the fine-tuned model
HF_MODEL_DIR = f'{VITS_DIR}/hf_model'
tts = MMS_TTS(lang='swh', model_dir=HF_MODEL_DIR, device='cuda')

# Test sentences
test_sentences = [
    'Habari, hali yako?',
    'Tanzania ni nchi nzuri sana.',
    'Karibu sana, asante kwa kuja.',
    'Sisi ni watu wa Kenya na tunapenda amani.',
    'Elimu ni ufunguo wa maisha.',
]

for text in test_sentences:
    waveform, sr = tts.synthesize(text)
    print(f'\n"{text}" ({len(waveform)/sr:.2f}s)')
    ipd.display(ipd.Audio(waveform, rate=sr))

In [ ]:
# Compare with the base (non-fine-tuned) model
tts_base = MMS_TTS(lang='swh', model_dir=None, device='cuda')

test_text = 'Habari za asubuhi, karibu Tanzania.'

wav_base, sr_base = tts_base.synthesize(test_text)
wav_fine, sr_fine = tts.synthesize(test_text)

print(f'Base model ({len(wav_base)/sr_base:.2f}s):')
ipd.display(ipd.Audio(wav_base, rate=sr_base))
print(f'\nFine-tuned ({len(wav_fine)/sr_fine:.2f}s):')
ipd.display(ipd.Audio(wav_fine, rate=sr_fine))

## 5. Verify Output Artifacts

Make sure everything is saved to Drive for the next steps.

In [ ]:
import os

print('=== VITS Output Artifacts ===')
print(f'\nDirectory: {VITS_DIR}')

# Check key files
key_files = [
    'train.csv', 'val.csv', 'speaker_map.json',
    'best_model.pth', 'final_model.pth',
    'hf_model/config.json', 'hf_model/model.safetensors',
]

for f in key_files:
    full_path = os.path.join(VITS_DIR, f)
    if os.path.exists(full_path):
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f'  [OK] {f} ({size_mb:.1f} MB)')
    else:
        print(f'  [MISSING] {f}')

# Count checkpoints
ckpts = [f for f in os.listdir(VITS_DIR) if f.startswith('checkpoint_')]
print(f'\nCheckpoints: {len(ckpts)}')
for c in sorted(ckpts):
    print(f'  {c}')

print('\n=== Next Step ===')
print('Run notebook 00b_data_pipeline.ipynb to:')
print('  1. Transcribe source audio with Whisper')
print('  2. Translate with MADLAD')
print('  3. Synthesize target speech using this VITS model')
print('  4. Compute alignments + silence insertion')